# AB Dor with amigo 0.1
AB Dor and its calibrator HD 37093 (JWST program 1093) from calslopes to a binary fit, with `amigo-pipeline` and
Virgil. Run with the venv `/fred/oz440/max/venvs/amigo-0.1` on a GPU.

In [ ]:
import time
from pathlib import Path

import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import pandas as pd
import dLux.utils as dlu
from IPython.display import Image, display

import amigo.files
from amigo.core_models import AmigoModel
from amigo.discos import to_virgil
from amigo.model_fits import SplineVisFit
from amigo.optical_models import AMIOptics
from amigo.vis_models import LogVisModel
from amigo_pipeline import Pipeline
from amigo_pipeline.calibration import load_calibration
from amigo_pipeline.dataset import DEFAULT_OPTICS, visibility_key
from virgil.pipeline import BinaryPipeline

from pipeline_memory import clear_compiled_between_stages, one_column_at_a_time

## Data
Calslopes made with `amigo.pipelines.process_calslope`.

In [ ]:
data_dir = "/fred/oz440/max/data/JWST/AB-DOR/calslope_amigo-0.1/"
files = amigo.files.get_files(data_dir, "calslope", EXP_TYPE="NIS_AMI", TARGPROP=["AB-DOR", "HD-37093"])
files = sorted(files, key=lambda f: f.filename())

## Instrument calibration
The paper's `calibration.npy` (amigo 0.0.10), converted by `convert_calibration_v2.py`.

In [ ]:
calibration = Path("/fred/oz440/max/data/amigo_files/v_0.0.11/v2_from_calibration_npy")
(detector, ramp, read), calibrated, defocus = load_calibration(calibration)

## Model
Exposures of one star, filter and roll share their visibilities.

In [ ]:
groups = {}
exposures = []
for f in files:
    role = "calibrator" if f[0].header["IS_PSF"] else "science"
    vis_key = visibility_key(f[0].header, role, groups, 1e-3)
    exposures.append(SplineVisFit(f, use_cov=True, only_diag=False, joint_fit=True, vis_key=vis_key))

filters = sorted({exp.filter for exp in exposures})
optics = AMIOptics(calibrated=calibrated, filters=filters, **DEFAULT_OPTICS)
placeholder = LogVisModel(dict(otf_coords=dlu.pixel_coords(3, 2 * optics.diameter),
                               modes={f: jnp.ones((1, 4), dtype=complex) / 2 for f in filters}))
model = AmigoModel(exposures, optics, detector, ramp, read, vis_model=placeholder)
for f in filters:
    model = model.set(f"params.defocus.{f}", jnp.asarray(defocus[f]))

## Pairs
Each AB Dor visit is calibrated by every HD 37093 exposure in its filter.

In [ ]:
pairs = []
for f in filters:
    calibrators = sorted(exp.key for exp in exposures if exp.filter == f and exp.calibrator)
    visits = {}
    for exp in exposures:
        if exp.filter == f and not exp.calibrator:
            visits.setdefault(exp.vis_key, []).append(exp)
    for visit in visits.values():
        pairs.append(dict(name=f"{visit[0].star}_{f}_obs{visit[0].observation}",
                          science=[exp.key for exp in visit], calibrator=calibrators, reference=calibrators[0]))

## Run the pipeline

In [ ]:
fits, null = one_column_at_a_time(model, exposures)
output = Path(f"/fred/oz440/max/outputs/retrain/pipeline/{time.strftime('%y-%m-%d_%H-%M-%S')}_ab-dor")
pipeline = Pipeline(exposures, model, pairs, output=output, fits=fits, null=null)
clear_compiled_between_stages(pipeline)
discos, cross = pipeline.run()

The pipeline draws its own figures. The calibrator's instrument fit, AB Dor fitted as a point source (the
companion shows in the residuals), and the DISCOs:

In [ ]:
for figure in ["instrument/pixels_0.png", "reference/pixels_0.png", "products/discos.png"]:
    display(Image(filename=str(output / figure)))

## Binary fit
Virgil's binary pipeline on each product. The 0.0.10 pipeline gave 332.6 ± 0.8 mas and PA 280.55 ± 0.13°.

In [ ]:
results = {name: BinaryPipeline(to_virgil(disco), output=output / "binary" / name, max_sep_mas=600.0, n_flux=80).run()
           for name, disco in discos.items()}

columns = ["sep_mas", "sep_err_mas", "pa_deg", "pa_err_deg", "flux", "flux_err", "delta_mag"]
pd.DataFrame({name: res.summary["companion"] for name, res in results.items()}).T[columns]

In [ ]:
plots = output / "binary" / next(iter(results)) / "plots"
for plot in ["search_delta_chi2", "limits_contrast_curve", "posterior_corner", "fit_correlation"]:
    display(Image(filename=str(plots / f"{plot}.png")))